In [1]:
"""
=============================================================================
German Credit Dataset — Guardrail Framework Transferability Validation
=============================================================================

Paper: "Counterfactual Explanation (CFE)-Based Risk Guardrail Design
        for Adverse Selection Prevention and Consumer Protection
        in AI Credit Scoring Models"

목적:
    HELOC 데이터셋에서 검증된 risk guardrail 프레임워크(Governed-DiCE)의
    이식 가능성(transferability)을 German Credit 데이터셋으로 검증합니다.

데이터:
    UCI German Credit Dataset (numeric version)
    - 파일: german.data-numeric (공백 구분자, 헤더 없음)
    - 샘플: 1,000개 | 피처: 24개 | 타겟: 마지막 컬럼 (1=Good, 2=Bad)
    - 출처: https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data

HELOC 대비 주요 변경사항:
    - 타겟: 1(Good)→1, 2(Bad)→0 재인코딩
    - Feature 분류: German Credit 도메인 기준으로 재설계
    - 하이퍼파라미터: Optuna 재탐색 (데이터 규모·분포 상이)
    - 인과 제약(Ωcau): German Credit 변수 간 관계로 재정의
    - 서브그룹 기준: Credit_Amount 사분위 (ExternalRiskEstimate 대응)
    - ±20% threshold 동일 적용 (transferability 검증 목적)

논문 기여:
    - 단일 데이터셋 의존성 한계 보완 (리뷰어 지적 대응)
    - guardrail 로직의 dataset-agnostic 이식성 실증
=============================================================================
"""

import pandas as pd
import numpy as np
import xgboost as xgb
import dice_ml
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix)
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from tqdm import tqdm
import warnings
import json
import time
from datetime import datetime

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------------------------
# Configuration (HELOC과 동일 설정 유지)
# ---------------------------------------------------------------------------
TOTAL_CFS    = 4
RANDOM_SEED  = 42
THRESHOLD    = 0.20
TEST_SIZE    = 0.2

# =============================================================================
# Part 1: Data Loading & Preprocessing
# =============================================================================
print("=" * 70)
print("PART 1: Data Loading & Preprocessing (German Credit)")
print("=" * 70)

# german.data-numeric: 공백 구분, 헤더 없음, 25컬럼
# 컬럼 정의 출처: german.doc (UCI Repository)
col_names = [
    'Checking_Account',      # 1: checking account status (categorical)
    'Duration',              # 2: duration in months (numerical)
    'Credit_History',        # 3: credit history (categorical)
    'Purpose',               # 4: purpose (categorical)
    'Credit_Amount',         # 5: credit amount (numerical)
    'Savings_Account',       # 6: savings account/bonds (categorical)
    'Employment',            # 7: present employment since (categorical)
    'Installment_Rate',      # 8: installment rate in % of disposable income
    'Personal_Status',       # 9: personal status and sex (categorical)
    'Other_Debtors',         # 10: other debtors/guarantors (categorical)
    'Residence_Since',       # 11: present residence since
    'Property',              # 12: property (categorical)
    'Age',                   # 13: age in years
    'Other_Installments',    # 14: other installment plans (categorical)
    'Housing',               # 15: housing (categorical)
    'Existing_Credits',      # 16: number of existing credits at this bank
    'Job',                   # 17: job (categorical)
    'Num_Dependents',        # 18: number of people liable for maintenance
    'Telephone',             # 19: telephone (binary)
    'Foreign_Worker',        # 20: foreign worker (binary)
    # One-hot encoded purpose columns (21-24): domestic appliances,
    # repairs, other, not specified
    'Purpose_A',
    'Purpose_B',
    'Purpose_C',
    'Purpose_D',
    'Risk'                   # 25: target (1=Good, 2=Bad)
]

df_raw = pd.read_csv('german.data-numeric', sep=r'\s+', header=None,
                     names=col_names)

print(f"Raw data loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} cols")
print(f"Target distribution (raw): {df_raw['Risk'].value_counts().to_dict()}")

# 타겟 재인코딩: 1(Good)→1, 2(Bad)→0
df_raw['Risk'] = df_raw['Risk'].map({1: 1, 2: 0})
print(f"Target after recode: Good(1)={df_raw['Risk'].sum()}, "
      f"Bad(0)={(df_raw['Risk']==0).sum()}")

target = 'Risk'
X = df_raw.drop(target, axis=1)
y = df_raw[target]

# dataset 변수: DiCE용 전체 DataFrame
dataset = df_raw.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

print(f"\nTotal samples : {len(df_raw):,}")
print(f"Training set  : {len(X_train):,}")
print(f"Test set      : {len(X_test):,}")

# 주요 변수 기술통계
desc_vars = ['Credit_Amount', 'Duration', 'Age', 'Installment_Rate']
desc_stats = X[desc_vars].describe().T[['mean', 'std', 'min', 'max']]
desc_stats.columns = ['Mean', 'Std', 'Min', 'Max']
print("\n[Descriptive Statistics - Key Variables]")
print(desc_stats.round(2))

# =============================================================================
# Part 2: XGBoost Model Training (Optuna)
# =============================================================================
print("\n" + "=" * 70)
print("PART 2: XGBoost Hyperparameter Optimization (Optuna, 100 Trials)")
print("=" * 70)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'random_state': RANDOM_SEED,
        'eval_metric': 'logloss',
        'use_label_encoder': False
    }
    mdl = xgb.XGBClassifier(**params)
    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    return cross_val_score(mdl, X_train, y_train, cv=cv,
                           scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize',
                            study_name='xgb_german_credit')
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = study.best_params
best_params.update({'random_state': RANDOM_SEED,
                    'eval_metric': 'logloss',
                    'use_label_encoder': False})

print(f"\nBest 5-Fold CV AUC: {study.best_value:.4f}")
print(f"Best Hyperparameters:\n{json.dumps(best_params, indent=2)}")

model = xgb.XGBClassifier(**best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, y_pred)

perf_table = pd.DataFrame({
    'Metric': ['Accuracy', 'AUC-ROC', 'F1-Score', 'Precision', 'Recall',
               '5-Fold CV AUC (Train)'],
    'Value': [f"{accuracy_score(y_test,y_pred):.4f}",
              f"{roc_auc_score(y_test,y_prob):.4f}",
              f"{f1_score(y_test,y_pred):.4f}",
              f"{precision_score(y_test,y_pred):.4f}",
              f"{recall_score(y_test,y_pred):.4f}",
              f"{study.best_value:.4f}"]
})
print("\n[Model Performance (Test Set)]")
print(perf_table.to_string(index=False))
print(f"Confusion Matrix: TN={cm[0][0]} FP={cm[0][1]} "
      f"FN={cm[1][0]} TP={cm[1][1]}")

perf_table.to_csv('german_model_performance.csv', index=False)

# =============================================================================
# Part 3: Feature Classification & Guardrail Configuration
# =============================================================================
print("\n" + "=" * 70)
print("PART 3: Feature Classification & Guardrail Configuration")
print("=" * 70)

# ── Immutable Features (Ωimm) ─────────────────────────────────────────────
# 차용인이 단기간에 변경 불가능한 역사적 사실 또는 외부 지정 지표
immutable_features = [
    'Age',              # 나이: 변경 불가
    'Foreign_Worker',   # 외국인 근로자 여부: 변경 불가
    'Personal_Status',  # 성별/결혼상태: 변경 불가
    'Credit_History',   # 과거 신용이력: 역사적 사실
    'Residence_Since',  # 현거주지 거주 기간: 역사적 사실
]

# ── Actionable Features (Ωact) ────────────────────────────────────────────
# 6개월 내 행동 변화로 조정 가능한 지표
actionable_features = [
    'Duration',          # 대출 기간: 재협상 가능
    'Installment_Rate',  # 할부 비율: 상환 계획 조정 가능
    'Existing_Credits',  # 현재 신용 건수: 상환 후 감소 가능
    'Savings_Account',   # 저축 계좌 잔액: 저축 증대 가능
    'Num_Dependents',    # 부양가족 수: 직접 변경 불가하나 신고 수정 가능
]

# ── Non-controllable (단기 변경 불가, 비불변) ──────────────────────────────
non_controllable_features = [
    'Credit_Amount',     # 대출 금액: 신청 시 결정
    'Purpose',           # 대출 목적: 신청 시 결정
    'Employment',        # 고용 기간: 단기 변경 어려움
    'Other_Debtors',     # 보증인/공동채무자: 단기 변경 어려움
    'Property',          # 자산: 단기 변경 어려움
    'Other_Installments',# 기타 할부: 단기 변경 어려움
    'Housing',           # 주거 형태: 단기 변경 어려움
    'Job',               # 직업 유형: 단기 변경 어려움
]

feature_classification = {}
for f in X.columns:
    if f in immutable_features:
        feature_classification[f] = 'Immutable'
    elif f in actionable_features:
        feature_classification[f] = 'Actionable'
    elif f in non_controllable_features:
        feature_classification[f] = 'Non-controllable'
    else:
        feature_classification[f] = 'Other'

print("[Feature Classification]")
for cat in ['Immutable', 'Actionable', 'Non-controllable', 'Other']:
    feats = [f for f, c in feature_classification.items() if c == cat]
    print(f"  {cat:<18}: {feats}")

# ── DiCE Setup ────────────────────────────────────────────────────────────
continuous_features = list(X.columns)

d_dice    = dice_ml.Data(dataframe=dataset,
                         continuous_features=continuous_features,
                         outcome_name=target)
m_dice    = dice_ml.Model(model=model, backend="sklearn")
exp_random = dice_ml.Dice(d_dice, m_dice, method="random")

rejected_all = X_test[model.predict(X_test) == 0].copy()
print(f"\nRejected borrowers (test set): {len(rejected_all):,}")

# =============================================================================
# Utility Functions
# =============================================================================

def build_permitted_range(query_row, features, threshold):
    pr = {}
    for f in features:
        val = query_row[f].values[0]
        lo  = min(val * (1 - threshold), val * (1 + threshold))
        hi  = max(val * (1 - threshold), val * (1 + threshold))
        if abs(lo - hi) < 0.02:
            lo -= 0.5
            hi += 0.5
        pr[f] = [lo, hi]
    return pr


def check_causal_violations_german(orig_dict, cf_dict):
    """
    German Credit 도메인 인과 제약 (Ωcau)

    Rule 1 - Duration-Installment 방향 일관성:
        대출 기간(Duration)이 단축되면 월 할부액 비율(Installment_Rate)이
        동시에 감소해서는 안 됨.
        (기간 단축 + 할부율 감소 = 분모 조작을 통한 외형 개선)

    Rule 2 - Savings-Existing_Credits 방향 일관성:
        저축 계좌 등급(Savings_Account)이 개선되면서
        보유 신용 건수(Existing_Credits)가 증가하는 경우,
        이는 실질 신용 개선이 아닌 지표 조작 가능성이 있음.
        → 저축 개선 시 신용 건수는 유지 또는 감소해야 함.
    """
    violations = 0

    # Rule 1: Duration 감소 시 Installment_Rate도 동시 감소 불허
    dur_orig = orig_dict.get('Duration', 0)
    dur_cf   = cf_dict.get('Duration', dur_orig)
    inst_orig = orig_dict.get('Installment_Rate', 0)
    inst_cf   = cf_dict.get('Installment_Rate', inst_orig)
    if dur_cf < dur_orig and inst_cf < inst_orig:
        violations += 1

    # Rule 2: Savings 개선 시 Existing_Credits 증가 불허
    sav_orig = orig_dict.get('Savings_Account', 0)
    sav_cf   = cf_dict.get('Savings_Account', sav_orig)
    cred_orig = orig_dict.get('Existing_Credits', 0)
    cred_cf   = cf_dict.get('Existing_Credits', cred_orig)
    if sav_cf > sav_orig and cred_cf > cred_orig:
        violations += 1

    return violations


def analyze_cf_paths(query_row, cf_df, target_col, imm_feats, act_feats):
    result = {
        'total_paths': 0, 'success_paths': 0,
        'imm_violation_paths': 0, 'cau_violation_paths': 0,
        'feat_changes_list': []
    }
    if cf_df is None or cf_df.empty:
        return result

    orig = query_row.iloc[0]
    for i in range(len(cf_df)):
        cf_row = cf_df.iloc[i]
        if target_col not in cf_df.columns or pd.isna(cf_row[target_col]):
            continue
        result['total_paths'] += 1
        if int(cf_row[target_col]) == 1:
            result['success_paths'] += 1
            if any(abs(cf_row[f] - orig[f]) > 1e-5
                   for f in imm_feats if f in cf_df.columns):
                result['imm_violation_paths'] += 1
            if check_causal_violations_german(
                    orig.to_dict(), cf_row.to_dict()) > 0:
                result['cau_violation_paths'] += 1
            check_feats = act_feats if act_feats else [
                c for c in X.columns if c in cf_df.columns]
            result['feat_changes_list'].append(
                sum(abs(cf_row[f] - orig[f]) > 1e-5
                    for f in check_feats if f in cf_df.columns))
    return result


# =============================================================================
# Part 4: Scenario Simulation (A / B / C)
# =============================================================================
print("\n" + "=" * 70)
print("PART 4: Scenario Simulation (A / B / C)")
print("=" * 70)

def run_scenario(scenario_name, rejected_df, exp_dice_obj, threshold=0.20):
    n = len(rejected_df)
    sample_success = 0
    success_paths  = 0
    imm_violations = 0
    cau_violations = 0
    feat_changes   = []
    generated_paths = 0

    print(f"\n--- {scenario_name} (N={n}) ---")
    for i in tqdm(range(n), desc=scenario_name):
        query = rejected_df.iloc[i:i + 1]

        if scenario_name == "Scenario_A":
            ftv, pr = "all", None
        elif scenario_name == "Scenario_B":
            ftv = [c for c in X.columns if c not in immutable_features]
            pr  = None
        elif scenario_name == "Scenario_C":
            ftv = actionable_features
            pr  = build_permitted_range(query, actionable_features, threshold)

        try:
            res = exp_dice_obj.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=ftv, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=RANDOM_SEED
            )
            cf_df = res.cf_examples_list[0].final_cfs_df
            act_f = actionable_features if scenario_name == "Scenario_C" \
                    else None
            r = analyze_cf_paths(query, cf_df, target,
                                 immutable_features, act_f)

            generated_paths += r['total_paths']
            success_paths   += r['success_paths']
            imm_violations  += r['imm_violation_paths']
            cau_violations  += r['cau_violation_paths']
            feat_changes.extend(r['feat_changes_list'])
            if r['success_paths'] > 0:
                sample_success += 1
        except Exception:
            continue

    rr     = sample_success / n if n > 0 else 0
    vr     = imm_violations / success_paths if success_paths > 0 else 0
    cau_vr = cau_violations / success_paths if success_paths > 0 else 0
    rrr    = rr * (1 - vr) * (1 - cau_vr)
    avg_ch = np.mean(feat_changes) if feat_changes else 0

    row = {
        'Scenario': scenario_name, 'N_Samples': n,
        'Sample_Success': sample_success,
        'RR(%)': round(rr * 100, 2),
        'Total_Paths_Generated': generated_paths,
        'Success_Paths': success_paths,
        'Imm_Violation_Paths': imm_violations,
        'VR(%)': round(vr * 100, 2),
        'Cau_Violation_Paths': cau_violations,
        'Causal_VR(%)': round(cau_vr * 100, 2),
        'Reliable_RR(%)': round(rrr * 100, 2),
        'Avg_Features_Changed': round(avg_ch, 2)
    }
    print(f"  RR={row['RR(%)']:.2f}% | VR={row['VR(%)']:.2f}% | "
          f"Causal_VR={row['Causal_VR(%)']:.2f}% | "
          f"Reliable_RR={row['Reliable_RR(%)']:.2f}%")
    return row

start = time.time()
scenario_rows = []
for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
    scenario_rows.append(
        run_scenario(sc, rejected_all, exp_random, THRESHOLD))
print(f"\n[Elapsed] {(time.time()-start)/60:.1f} min")

df_scenarios = pd.DataFrame(scenario_rows)
print("\n[Scenario Comparison — German Credit]")
print(df_scenarios[['Scenario','N_Samples','RR(%)','VR(%)',
                     'Causal_VR(%)','Reliable_RR(%)',
                     'Avg_Features_Changed']].to_string(index=False))
df_scenarios.to_csv('german_scenario_comparison.csv', index=False)

# =============================================================================
# Part 5: Sensitivity Analysis (±10% ~ ±30%)
# =============================================================================
print("\n" + "=" * 70)
print("PART 5: Sensitivity Analysis")
print("=" * 70)

sensitivity_rows = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30]:
    label = f"±{int(t*100)}%"
    n = len(rejected_all)
    s_success, s_paths, i_viol, c_viol = 0, 0, 0, 0
    f_changes = []

    for i in tqdm(range(n), desc=label):
        query = rejected_all.iloc[i:i+1]
        pr    = build_permitted_range(query, actionable_features, t)
        try:
            res = exp_random.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=actionable_features, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=RANDOM_SEED)
            cf_df = res.cf_examples_list[0].final_cfs_df
            r = analyze_cf_paths(query, cf_df, target,
                                 immutable_features, actionable_features)
            s_paths  += r['success_paths']
            i_viol   += r['imm_violation_paths']
            c_viol   += r['cau_violation_paths']
            f_changes.extend(r['feat_changes_list'])
            if r['success_paths'] > 0:
                s_success += 1
        except Exception:
            continue

    rr  = s_success / n if n > 0 else 0
    vr  = i_viol / s_paths if s_paths > 0 else 0
    cvr = c_viol / s_paths if s_paths > 0 else 0
    sensitivity_rows.append({
        'Threshold': label, 'N': n,
        'RR(%)': round(rr*100, 2),
        'VR(%)': round(vr*100, 2),
        'Causal_VR(%)': round(cvr*100, 2),
        'Reliable_RR(%)': round(rr*(1-vr)*(1-cvr)*100, 2),
        'Avg_Features_Changed': round(np.mean(f_changes) if f_changes else 0, 2)
    })

df_sens = pd.DataFrame(sensitivity_rows)
print(df_sens.to_string(index=False))
df_sens.to_csv('german_sensitivity_analysis.csv', index=False)

# =============================================================================
# Part 6 (수정): Subgroup Analysis (Duration Quartiles)
# =============================================================================
# 수정 이유:
#   Credit_Amount는 범주 코드값(1~5)이라 사분위 분할이 무의미함
#   → Q2, Q3가 N=0으로 붕괴됨
#   German Credit의 연속형 수치 변수 중 가장 신용 위험과 직결되는
#   Duration(대출 기간, 4~72개월)으로 교체
#   HELOC의 ExternalRiskEstimate와 유사하게:
#     낮은 Duration(Q1) = 단기 소액 → 상환 부담 낮음 → recourse 용이
#     높은 Duration(Q4) = 장기 대출 → 상환 부담 높음 → recourse 어려움
# =============================================================================

print("\n" + "=" * 70)
print("PART 6 (수정): Subgroup Analysis (Duration Quartiles)")
print("=" * 70)

# Duration 분포 확인
print(f"\nDuration stats (rejected borrowers):")
print(f"  Min={rejected_all['Duration'].min():.0f}  "
      f"Max={rejected_all['Duration'].max():.0f}  "
      f"Mean={rejected_all['Duration'].mean():.1f}  "
      f"Median={rejected_all['Duration'].median():.1f}")

dur = rejected_all['Duration']
q_bounds = dur.quantile([0.25, 0.50, 0.75])
q1_t = q_bounds.iloc[0]
q2_t = q_bounds.iloc[1]
q3_t = q_bounds.iloc[2]
print(f"Quartile boundaries: Q1≤{q1_t:.0f}mo, Q2≤{q2_t:.0f}mo, "
      f"Q3≤{q3_t:.0f}mo")

# 중복 경계 방지: 경계가 같으면 고유값 기반으로 분할
unique_vals = sorted(dur.unique())
if len(set([q1_t, q2_t, q3_t])) < 3:
    # 고유값이 4개 미만이면 중앙값 기준 이분
    med = dur.median()
    subgroups = {
        'Q1~Q2 (Short, ≤median)': rejected_all[dur <= med],
        'Q3~Q4 (Long, >median)':  rejected_all[dur >  med],
    }
    print(f"  ※ Quartile collapse detected → binary split at median "
          f"({med:.0f} months)")
else:
    subgroups = {
        'Q1 (Short, ≤{:.0f}mo)'.format(q1_t):
            rejected_all[dur <= q1_t],
        'Q2 ({:.0f}~{:.0f}mo)'.format(q1_t, q2_t):
            rejected_all[(dur > q1_t) & (dur <= q2_t)],
        'Q3 ({:.0f}~{:.0f}mo)'.format(q2_t, q3_t):
            rejected_all[(dur > q2_t) & (dur <= q3_t)],
        'Q4 (Long, >{:.0f}mo)'.format(q3_t):
            rejected_all[dur > q3_t],
    }

# 빈 그룹 제거
subgroups = {k: v for k, v in subgroups.items() if len(v) > 0}
print(f"\nSubgroup sizes: "
      + " | ".join(f"{k}=N{len(v)}" for k, v in subgroups.items()))

subgroup_rows = []
for grp_name, grp_df in subgroups.items():
    n_grp = len(grp_df)
    print(f"\n--- {grp_name} (N={n_grp}) ---")
    s_success, s_paths, i_viol, c_viol, f_changes = 0, 0, 0, 0, []

    for i in tqdm(range(n_grp), desc=grp_name):
        query = grp_df.iloc[i:i+1]
        pr    = build_permitted_range(query, actionable_features, THRESHOLD)
        try:
            res = exp_random.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=actionable_features, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=RANDOM_SEED)
            cf_df = res.cf_examples_list[0].final_cfs_df
            r = analyze_cf_paths(query, cf_df, target,
                                 immutable_features, actionable_features)
            s_paths  += r['success_paths']
            i_viol   += r['imm_violation_paths']
            c_viol   += r['cau_violation_paths']
            f_changes.extend(r['feat_changes_list'])
            if r['success_paths'] > 0:
                s_success += 1
        except Exception:
            continue

    rr  = s_success / n_grp if n_grp > 0 else 0
    vr  = i_viol / s_paths  if s_paths > 0 else 0
    cvr = c_viol / s_paths  if s_paths > 0 else 0
    rrr = rr * (1 - vr) * (1 - cvr)
    avg_ch = np.mean(f_changes) if f_changes else 0

    # Duration 평균 (그룹 특성 요약)
    dur_mean = grp_df['Duration'].mean()

    subgroup_rows.append({
        'Subgroup':             grp_name,
        'N':                    n_grp,
        'Avg_Duration(mo)':     round(dur_mean, 1),
        'Sample_Success':       s_success,
        'RR(%)':                round(rr  * 100, 2),
        'Reliable_RR(%)':       round(rrr * 100, 2),
        'Avg_Features_Changed': round(avg_ch, 2),
    })

df_sub = pd.DataFrame(subgroup_rows)

print("\n[Subgroup Analysis — German Credit (Duration Quartiles)]")
print(df_sub.to_string(index=False))
df_sub.to_csv('german_subgroup_analysis.csv', index=False)

# ---------------------------------------------------------------------------
# 시각화: Duration 사분위별 Reliable RR
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
colors_sub = ['#5B9BD5', '#70AD47', '#ED7D31', '#C00000'][:len(df_sub)]
bars = ax.bar(range(len(df_sub)),
              df_sub['Reliable_RR(%)'],
              color=colors_sub, edgecolor='black', linewidth=0.8)

for bar, val in zip(bars, df_sub['Reliable_RR(%)']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_xticks(range(len(df_sub)))
ax.set_xticklabels(df_sub['Subgroup'], fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Reliable RR (%)', fontsize=11)
ax.set_ylim(0, max(df_sub['Reliable_RR(%)']) * 1.25)
ax.set_title(
    'Recourse Accessibility by Loan Duration Quartile\n'
    '(German Credit Dataset, Scenario C, Full Guardrails)',
    fontsize=11, fontweight='bold'
)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('german_subgroup_duration.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n  Saved: german_subgroup_duration.png")

# ---------------------------------------------------------------------------
# 논문 삽입용 요약
# ---------------------------------------------------------------------------
print("\n[Paper-Ready Summary — German Credit Subgroup]")
if len(df_sub) >= 2:
    best  = df_sub.loc[df_sub['Reliable_RR(%)'].idxmax()]
    worst = df_sub.loc[df_sub['Reliable_RR(%)'].idxmin()]
    ratio = (best['Reliable_RR(%)'] / worst['Reliable_RR(%)']
             if worst['Reliable_RR(%)'] > 0 else float('inf'))
    print(f"""
Subgroup analysis stratifying {len(rejected_all)} rejected borrowers
by loan Duration (months) reveals a {ratio:.1f}-fold disparity in
Reliable RR between the most and least accessible groups:

  Best  ({best['Subgroup']:30s}): Reliable RR = {best['Reliable_RR(%)']:.2f}%
  Worst ({worst['Subgroup']:30s}): Reliable RR = {worst['Reliable_RR(%)']:.2f}%

This pattern is consistent with the HELOC finding that uniform ±20%
actionability thresholds encode structurally differential recourse
access across borrower segments, independent of dataset or jurisdiction.
""")

# =============================================================================
# Part 7: Representative Case Extraction (Vanilla vs Proposed)
# =============================================================================
print("\n" + "=" * 70)
print("PART 7: Representative Case Extraction (Vanilla vs Proposed)")
print("=" * 70)

cases = []
for i in range(min(100, len(rejected_all))):
    query = rejected_all.iloc[i:i+1]
    try:
        v_res = exp_random.generate_counterfactuals(
            query, total_CFs=TOTAL_CFS, desired_class="opposite",
            features_to_vary="all", proximity_weight=0.5,
            random_seed=RANDOM_SEED)
        v_cf = v_res.cf_examples_list[0].final_cfs_df

        pr    = build_permitted_range(query, actionable_features, THRESHOLD)
        p_res = exp_random.generate_counterfactuals(
            query, total_CFs=TOTAL_CFS, desired_class="opposite",
            features_to_vary=actionable_features, permitted_range=pr,
            proximity_weight=0.5, sparsity_weight=1.0,
            random_seed=RANDOM_SEED)
        p_cf = p_res.cf_examples_list[0].final_cfs_df

        if v_cf is None or v_cf.empty or p_cf is None or p_cf.empty:
            continue

        orig = query.iloc[0]
        for vi in range(len(v_cf)):
            v_row = v_cf.iloc[vi]
            if target in v_cf.columns and int(v_row[target]) == 1:
                has_viol = any(abs(v_row[f] - orig[f]) > 1e-5
                               for f in immutable_features
                               if f in v_cf.columns)
                if has_viol:
                    for pi in range(len(p_cf)):
                        p_row = p_cf.iloc[pi]
                        if target in p_cf.columns and int(p_row[target]) == 1:
                            cases.append({
                                'index': rejected_all.index[i],
                                'original': orig.to_dict(),
                                'vanilla_cf': v_row.to_dict(),
                                'proposed_cf': p_row.to_dict()
                            })
                            break
                    break
        if len(cases) >= 3:
            break
    except Exception:
        continue

if cases:
    key_vars = immutable_features[:3] + actionable_features[:3]
    for idx, case in enumerate(cases):
        print(f"\n[Case {idx+1}] (Test Index: {case['index']})")
        print(f"{'Variable':<25} {'Current':>10} {'Vanilla CF':>12}"
              f" {'Proposed CF':>12}")
        print("-" * 65)
        for v in key_vars:
            o   = case['original'].get(v, 0)
            van = case['vanilla_cf'].get(v, 0)
            prop = case['proposed_cf'].get(v, 0)
            flag = " ⚠" if (v in immutable_features and
                            abs(van - o) > 1e-5) else ""
            print(f"{v:<25} {o:>10.2f} {van:>12.2f} {prop:>12.2f}{flag}")
        print(f"{'[Approval]':<25} {'Reject':>10} {'Approve':>12}"
              f" {'Approve':>12}")
else:
    print("No representative cases found.")

# =============================================================================
# Part 8: Algorithm Robustness (Random vs KD-Tree)
# =============================================================================
print("\n" + "=" * 70)
print("PART 8: Algorithm Robustness Validation (Random vs KD-Tree)")
print("=" * 70)

algo_results = []
start = time.time()
for method in ["random", "kdtree"]:
    exp_obj = dice_ml.Dice(d_dice, m_dice, method=method)
    for scenario in ["Scenario_A", "Scenario_C"]:
        n = len(rejected_all)
        s_success, s_paths, i_viol, c_viol, f_changes, errors = \
            0, 0, 0, 0, [], 0

        for i in tqdm(range(n), desc=f"{scenario}×{method}"):
            query = rejected_all.iloc[i:i+1]
            ftv = "all" if scenario == "Scenario_A" else actionable_features
            pr  = None if scenario == "Scenario_A" else \
                  build_permitted_range(query, actionable_features, THRESHOLD)
            try:
                kw = dict(total_CFs=TOTAL_CFS, desired_class="opposite",
                          features_to_vary=ftv, permitted_range=pr,
                          proximity_weight=0.5, sparsity_weight=1.0)
                if method == "random":
                    kw['random_seed'] = RANDOM_SEED
                res   = exp_obj.generate_counterfactuals(query, **kw)
                cf_df = res.cf_examples_list[0].final_cfs_df
                act_f = actionable_features \
                        if scenario == "Scenario_C" else None
                r = analyze_cf_paths(query, cf_df, target,
                                     immutable_features, act_f)
                s_paths  += r['success_paths']
                i_viol   += r['imm_violation_paths']
                c_viol   += r['cau_violation_paths']
                f_changes.extend(r['feat_changes_list'])
                if r['success_paths'] > 0:
                    s_success += 1
            except Exception:
                errors += 1

        rr  = s_success / n if n > 0 else 0
        vr  = i_viol / s_paths  if s_paths > 0 else 0
        cvr = c_viol / s_paths  if s_paths > 0 else 0
        algo_results.append({
            'Scenario': scenario, 'Method': method, 'N': n,
            'RR(%)': round(rr*100, 2),
            'VR(%)': round(vr*100, 2),
            'Causal_VR(%)': round(cvr*100, 2),
            'Reliable_RR(%)': round(rr*(1-vr)*(1-cvr)*100, 2),
            'Avg_Features_Changed': round(
                np.mean(f_changes) if f_changes else 0, 2),
            'Errors': errors
        })

print(f"\n[Elapsed] {(time.time()-start)/60:.1f} min")
df_algo = pd.DataFrame(algo_results)
print(df_algo.to_string(index=False))
df_algo.to_csv('german_method_comparison.csv', index=False)

# =============================================================================
# Part 9: HELOC vs German Credit — Transferability Comparison Table
# =============================================================================
print("\n" + "=" * 70)
print("PART 9: Transferability Summary — HELOC vs German Credit")
print("=" * 70)

# HELOC 결과 (논문 기준값 하드코딩)
heloc_ref = pd.DataFrame([
    {'Dataset': 'HELOC', 'Scenario': 'Scenario_A',
     'RR(%)': 100.00, 'VR(%)': 94.11,
     'Causal_VR(%)': 11.49, 'Reliable_RR(%)': 5.22},
    {'Dataset': 'HELOC', 'Scenario': 'Scenario_B',
     'RR(%)': 36.87, 'VR(%)': 0.00,
     'Causal_VR(%)': 28.22, 'Reliable_RR(%)': 26.47},
    {'Dataset': 'HELOC', 'Scenario': 'Scenario_C',
     'RR(%)': 11.05, 'VR(%)': 0.00,
     'Causal_VR(%)': 46.92, 'Reliable_RR(%)': 5.87},
])

german_ref = df_scenarios[
    ['Scenario','RR(%)','VR(%)','Causal_VR(%)','Reliable_RR(%)']
].copy()
german_ref.insert(0, 'Dataset', 'German Credit')

transfer_table = pd.concat([heloc_ref, german_ref], ignore_index=True)
print(transfer_table.to_string(index=False))
transfer_table.to_csv('german_transferability_comparison.csv', index=False)

print("\n[Key Transferability Findings]")
for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
    h = heloc_ref[heloc_ref['Scenario'] == sc].iloc[0]
    g = german_ref[german_ref['Scenario'] == sc].iloc[0]
    vr_consistent = (h['VR(%)'] > 50) == (g['VR(%)'] > 50)
    b_improves_h  = heloc_ref[heloc_ref['Scenario']=='Scenario_B']\
                    ['Reliable_RR(%)'].values[0] > \
                    heloc_ref[heloc_ref['Scenario']=='Scenario_A']\
                    ['Reliable_RR(%)'].values[0]
    b_improves_g  = german_ref[german_ref['Scenario']=='Scenario_B']\
                    ['Reliable_RR(%)'].values[0] > \
                    german_ref[german_ref['Scenario']=='Scenario_A']\
                    ['Reliable_RR(%)'].values[0]
    print(f"  {sc}: HELOC Reliable_RR={h['Reliable_RR(%)']:.2f}% | "
          f"German Reliable_RR={g['Reliable_RR(%)']:.2f}%")

print(f"\n  Scenario B improvement over A:")
print(f"    HELOC  : {heloc_ref[heloc_ref['Scenario']=='Scenario_B']['Reliable_RR(%)'].values[0]:.2f}% "
      f"vs {heloc_ref[heloc_ref['Scenario']=='Scenario_A']['Reliable_RR(%)'].values[0]:.2f}%")
g_a = german_ref[german_ref['Scenario']=='Scenario_A']['Reliable_RR(%)'].values[0]
g_b = german_ref[german_ref['Scenario']=='Scenario_B']['Reliable_RR(%)'].values[0]
print(f"    German : {g_b:.2f}% vs {g_a:.2f}%")
fold = g_b / g_a if g_a > 0 else float('inf')
print(f"    → German Scenario B improvement: {fold:.1f}-fold")

# =============================================================================
# Summary
# =============================================================================
print("\n" + "=" * 70)
print(f"All experiments completed ({datetime.now().strftime('%Y-%m-%d %H:%M:%S')})")
print("=" * 70)
print("Output files:")
for f in ['german_model_performance.csv',
          'german_scenario_comparison.csv',
          'german_sensitivity_analysis.csv',
          'german_subgroup_analysis.csv',
          'german_method_comparison.csv',
          'german_transferability_comparison.csv']:
    print(f"  {f}")

C:\Users\ecredible\anaconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PART 1: Data Loading & Preprocessing (German Credit)
Raw data loaded: 1000 rows × 25 cols
Target distribution (raw): {1: 700, 2: 300}
Target after recode: Good(1)=700, Bad(0)=300

Total samples : 1,000
Training set  : 800
Test set      : 200

[Descriptive Statistics - Key Variables]
                   Mean    Std  Min   Max
Credit_Amount      2.10   1.58  1.0   5.0
Duration          20.90  12.06  4.0  72.0
Age                1.16   0.36  1.0   2.0
Installment_Rate   2.84   1.10  1.0   4.0

PART 2: XGBoost Hyperparameter Optimization (Optuna, 100 Trials)


Best trial: 93. Best value: 0.801339: 100%|██████████████████████████████████████████| 100/100 [01:13<00:00,  1.37it/s]



Best 5-Fold CV AUC: 0.8013
Best Hyperparameters:
{
  "n_estimators": 151,
  "max_depth": 3,
  "learning_rate": 0.039256736402625024,
  "subsample": 0.7777905462393178,
  "colsample_bytree": 0.621786971379521,
  "reg_alpha": 0.00044175049614219263,
  "reg_lambda": 2.9906799193653733e-06,
  "min_child_weight": 1,
  "gamma": 5.717632435590151e-06,
  "random_state": 42,
  "eval_metric": "logloss",
  "use_label_encoder": false
}

[Model Performance (Test Set)]
               Metric  Value
             Accuracy 0.7550
              AUC-ROC 0.7881
             F1-Score 0.8304
            Precision 0.8054
               Recall 0.8571
5-Fold CV AUC (Train) 0.8013
Confusion Matrix: TN=31 FP=29 FN=20 TP=120

PART 3: Feature Classification & Guardrail Configuration
[Feature Classification]
  Immutable         : ['Credit_History', 'Personal_Status', 'Residence_Since', 'Age', 'Foreign_Worker']
  Actionable        : ['Duration', 'Savings_Account', 'Installment_Rate', 'Existing_Credits', 'Num_Depende

Scenario_A: 100%|██████████████████████████████████████████████████████████████████████| 51/51 [00:13<00:00,  3.76it/s]


...

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Scenario_C×random:  92%|██████████████████████████████████████████████████████████     | 47/51 [00:18<00:01,  2.62it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Scenario_C×kdtree: 100%|███████████████████████████████████████████████████████████████| 51/51 [00:07<00:00,  6.51it/s]


[Elapsed] 0.9 min
  Scenario Method  N  RR(%)  VR(%)  Causal_VR(%)  Reliable_RR(%)  Avg_Features_Changed  Errors
Scenario_A random 51 100.00  24.51          0.49           75.12                  1.87       0
Scenario_C random 51  72.55   0.00         12.16           63.73                  2.03      14
Scenario_A kdtree 51 100.00  90.20          4.90            9.32                  8.76       0
Scenario_C kdtree 51   0.00   0.00          0.00            0.00                  0.00      51

PART 9: Transferability Summary — HELOC vs German Credit
      Dataset   Scenario  RR(%)  VR(%)  Causal_VR(%)  Reliable_RR(%)
        HELOC Scenario_A 100.00  94.11         11.49            5.22
        HELOC Scenario_B  36.87   0.00         28.22           26.47
        HELOC Scenario_C  11.05   0.00         46.92            5.87
German Credit Scenario_A 100.00  24.51          0.49           75.12
German Credit Scenario_B 100.00   0.00          0.00          100.00
German Credit Scenario_C  72.55   

In [2]:
# =============================================================================
# Part 6 (수정): Subgroup Analysis (Duration Quartiles)
# =============================================================================
# 수정 이유:
#   Credit_Amount는 범주 코드값(1~5)이라 사분위 분할이 무의미함
#   → Q2, Q3가 N=0으로 붕괴됨
#   German Credit의 연속형 수치 변수 중 가장 신용 위험과 직결되는
#   Duration(대출 기간, 4~72개월)으로 교체
#   HELOC의 ExternalRiskEstimate와 유사하게:
#     낮은 Duration(Q1) = 단기 소액 → 상환 부담 낮음 → recourse 용이
#     높은 Duration(Q4) = 장기 대출 → 상환 부담 높음 → recourse 어려움
# =============================================================================

print("\n" + "=" * 70)
print("PART 6 (수정): Subgroup Analysis (Duration Quartiles)")
print("=" * 70)

# Duration 분포 확인
print(f"\nDuration stats (rejected borrowers):")
print(f"  Min={rejected_all['Duration'].min():.0f}  "
      f"Max={rejected_all['Duration'].max():.0f}  "
      f"Mean={rejected_all['Duration'].mean():.1f}  "
      f"Median={rejected_all['Duration'].median():.1f}")

dur = rejected_all['Duration']
q_bounds = dur.quantile([0.25, 0.50, 0.75])
q1_t = q_bounds.iloc[0]
q2_t = q_bounds.iloc[1]
q3_t = q_bounds.iloc[2]
print(f"Quartile boundaries: Q1≤{q1_t:.0f}mo, Q2≤{q2_t:.0f}mo, "
      f"Q3≤{q3_t:.0f}mo")

# 중복 경계 방지: 경계가 같으면 고유값 기반으로 분할
unique_vals = sorted(dur.unique())
if len(set([q1_t, q2_t, q3_t])) < 3:
    # 고유값이 4개 미만이면 중앙값 기준 이분
    med = dur.median()
    subgroups = {
        'Q1~Q2 (Short, ≤median)': rejected_all[dur <= med],
        'Q3~Q4 (Long, >median)':  rejected_all[dur >  med],
    }
    print(f"  ※ Quartile collapse detected → binary split at median "
          f"({med:.0f} months)")
else:
    subgroups = {
        'Q1 (Short, ≤{:.0f}mo)'.format(q1_t):
            rejected_all[dur <= q1_t],
        'Q2 ({:.0f}~{:.0f}mo)'.format(q1_t, q2_t):
            rejected_all[(dur > q1_t) & (dur <= q2_t)],
        'Q3 ({:.0f}~{:.0f}mo)'.format(q2_t, q3_t):
            rejected_all[(dur > q2_t) & (dur <= q3_t)],
        'Q4 (Long, >{:.0f}mo)'.format(q3_t):
            rejected_all[dur > q3_t],
    }

# 빈 그룹 제거
subgroups = {k: v for k, v in subgroups.items() if len(v) > 0}
print(f"\nSubgroup sizes: "
      + " | ".join(f"{k}=N{len(v)}" for k, v in subgroups.items()))

subgroup_rows = []
for grp_name, grp_df in subgroups.items():
    n_grp = len(grp_df)
    print(f"\n--- {grp_name} (N={n_grp}) ---")
    s_success, s_paths, i_viol, c_viol, f_changes = 0, 0, 0, 0, []

    for i in tqdm(range(n_grp), desc=grp_name):
        query = grp_df.iloc[i:i+1]
        pr    = build_permitted_range(query, actionable_features, THRESHOLD)
        try:
            res = exp_random.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=actionable_features, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=RANDOM_SEED)
            cf_df = res.cf_examples_list[0].final_cfs_df
            r = analyze_cf_paths(query, cf_df, target,
                                 immutable_features, actionable_features)
            s_paths  += r['success_paths']
            i_viol   += r['imm_violation_paths']
            c_viol   += r['cau_violation_paths']
            f_changes.extend(r['feat_changes_list'])
            if r['success_paths'] > 0:
                s_success += 1
        except Exception:
            continue

    rr  = s_success / n_grp if n_grp > 0 else 0
    vr  = i_viol / s_paths  if s_paths > 0 else 0
    cvr = c_viol / s_paths  if s_paths > 0 else 0
    rrr = rr * (1 - vr) * (1 - cvr)
    avg_ch = np.mean(f_changes) if f_changes else 0

    # Duration 평균 (그룹 특성 요약)
    dur_mean = grp_df['Duration'].mean()

    subgroup_rows.append({
        'Subgroup':             grp_name,
        'N':                    n_grp,
        'Avg_Duration(mo)':     round(dur_mean, 1),
        'Sample_Success':       s_success,
        'RR(%)':                round(rr  * 100, 2),
        'Reliable_RR(%)':       round(rrr * 100, 2),
        'Avg_Features_Changed': round(avg_ch, 2),
    })

df_sub = pd.DataFrame(subgroup_rows)

print("\n[Subgroup Analysis — German Credit (Duration Quartiles)]")
print(df_sub.to_string(index=False))
df_sub.to_csv('german_subgroup_analysis.csv', index=False)

# ---------------------------------------------------------------------------
# 시각화: Duration 사분위별 Reliable RR
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
colors_sub = ['#5B9BD5', '#70AD47', '#ED7D31', '#C00000'][:len(df_sub)]
bars = ax.bar(range(len(df_sub)),
              df_sub['Reliable_RR(%)'],
              color=colors_sub, edgecolor='black', linewidth=0.8)

for bar, val in zip(bars, df_sub['Reliable_RR(%)']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_xticks(range(len(df_sub)))
ax.set_xticklabels(df_sub['Subgroup'], fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Reliable RR (%)', fontsize=11)
ax.set_ylim(0, max(df_sub['Reliable_RR(%)']) * 1.25)
ax.set_title(
    'Recourse Accessibility by Loan Duration Quartile\n'
    '(German Credit Dataset, Scenario C, Full Guardrails)',
    fontsize=11, fontweight='bold'
)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('german_subgroup_duration.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n  Saved: german_subgroup_duration.png")

# ---------------------------------------------------------------------------
# 논문 삽입용 요약
# ---------------------------------------------------------------------------
print("\n[Paper-Ready Summary — German Credit Subgroup]")
if len(df_sub) >= 2:
    best  = df_sub.loc[df_sub['Reliable_RR(%)'].idxmax()]
    worst = df_sub.loc[df_sub['Reliable_RR(%)'].idxmin()]
    ratio = (best['Reliable_RR(%)'] / worst['Reliable_RR(%)']
             if worst['Reliable_RR(%)'] > 0 else float('inf'))
    print(f"""
Subgroup analysis stratifying {len(rejected_all)} rejected borrowers
by loan Duration (months) reveals a {ratio:.1f}-fold disparity in
Reliable RR between the most and least accessible groups:

  Best  ({best['Subgroup']:30s}): Reliable RR = {best['Reliable_RR(%)']:.2f}%
  Worst ({worst['Subgroup']:30s}): Reliable RR = {worst['Reliable_RR(%)']:.2f}%

This pattern is consistent with the HELOC finding that uniform ±20%
actionability thresholds encode structurally differential recourse
access across borrower segments, independent of dataset or jurisdiction.
""")


PART 6 (수정): Subgroup Analysis (Duration Quartiles)

Duration stats (rejected borrowers):
  Min=9  Max=60  Mean=27.3  Median=24.0
Quartile boundaries: Q1≤18mo, Q2≤24mo, Q3≤36mo

Subgroup sizes: Q1 (Short, ≤18mo)=N16 | Q2 (18~24mo)=N14 | Q3 (24~36mo)=N14 | Q4 (Long, >36mo)=N7

--- Q1 (Short, ≤18mo) (N=16) ---


Q1 (Short, ≤18mo):  12%|████████                                                        | 2/16 [00:00<00:06,  2.32it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Q1 (Short, ≤18mo):  38%|████████████████████████                                        | 6/16 [00:02<00:04,  2.47it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


...


Q4 (Long, >36mo):  71%|███████████████████████████████████████████████▏                  | 5/7 [00:02<00:00,  2.24it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Q4 (Long, >36mo): 100%|██████████████████████████████████████████████████████████████████| 7/7 [00:02<00:00,  2.34it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec

[Subgroup Analysis — German Credit (Duration Quartiles)]
         Subgroup  N  Avg_Duration(mo)  Sample_Success  RR(%)  Reliable_RR(%)  Avg_Features_Changed
Q1 (Short, ≤18mo) 16              15.0              14  87.50           81.25                  1.93
     Q2 (18~24mo) 14              24.0              12  85.71           75.00                  2.10
     Q3 (24~36mo) 14              34.3               9  64.29           50.00                  2.14
 Q4 (Long, >36mo)  7              48.0               2  28.57           28.57                  1.75



  Saved: german_subgroup_duration.png

[Paper-Ready Summary — German Credit Subgroup]

Subgroup analysis stratifying 51 rejected borrowers
by loan Duration (months) reveals a 2.8-fold disparity in
Reliable RR between the most and least accessible groups:

  Best  (Q1 (Short, ≤18mo)             ): Reliable RR = 81.25%
  Worst (Q4 (Long, >36mo)              ): Reliable RR = 28.57%

This pattern is consistent with the HELOC finding that uniform ±20%
actionability thresholds encode structurally differential recourse
access across borrower segments, independent of dataset or jurisdiction.

